In [1]:
## IMPORTS

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
import os
import dask
import dask.array

import datetime
from datetime import date

from collections import Counter
from collections import defaultdict

import pystac_client
from pystac_client import Client
from pystac.extensions.projection import ProjectionExtension as proj
from pystac import ItemCollection

import planetary_computer
import rasterio
import rasterio.features
from rasterio.features import rasterize

import stackstac
import pyproj

import dask.diagnostics

from shapely.geometry import box
from shapely.ops import transform

from scipy.ndimage import binary_propagation
from scipy.ndimage import label



In [2]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


In [3]:
## FUNCTION TO DEFINE BOUNDING BOX AROUND A GIVEN CENTROID
def bounds_latlon_around(center_lon, center_lat, side_m=10000):
    """
    center_lon, center_lat : centroid in decimal degrees (EPSG:4326)
    side_m                 : length of box side in meters (default 10 km)
    returns                 : (minx, miny, maxx, maxy) in lon/lat
    """
    # 1) set up transformers
    to_ps = pyproj.Transformer.from_crs(4326, 3413, always_xy=True).transform
    to_ll = pyproj.Transformer.from_crs(3413, 4326, always_xy=True).transform

    # 2) project centroid into EPSG:3413 (units = m)
    x0, y0 = to_ps(center_lon, center_lat)

    # 3) build a square of side `side_m` centered on (x0,y0)
    half = side_m / 2.0
    sq_m = box(x0 - half, y0 - half, x0 + half, y0 + half)

    # 4) reproject that square back to lon/lat and grab its bounds
    sq_ll = transform(to_ll, sq_m)
    return sq_ll.bounds  # (minx, miny, maxx, maxy)




In [10]:
## APPLY FUNCTION TO GET BOUNDING BOX OF CERTAIN SIZE AROUND CENTROID OF INTEREST

gdf = gpd.read_file("/home/jupyter/repos/lake-vision/sandbox/dunmire_cw2019_shuttle.geojson")
centroid = (gdf.centroid_x.iloc[0], gdf.centroid_y.iloc[0])
# centroid = (-49.495, 68.725)       # lon, lat of North Lake, for example
# centroid = (-49.26457,68.63694)
bounds_latlon = bounds_latlon_around(*centroid, side_m=6000) # box side length
print(bounds_latlon)



(-48.512722608522544, 69.10023785143531, -48.35355345430137, 69.156993641399)


In [66]:
## GET COLLECTION FOR THE AREA OF INTEREST (BOX AROUND CENTROID) DEFINED ABOVE

time_range = '2019-05-01/2019-09-30'
bbox = bounds_latlon

band_names = ["B04",  # red (665 nm)
              "B03",  # green (560 nm)
              "B02",  # blue (490 nm)
              "B08",  # NIR (842 nm)
              "B11"]  # SWIR1 (1610 nm)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 100}}
)

items = search.item_collection()

print(f'There are {len(items)} images that fit your search criteria in the database.')
print(type(items))



There are 227 images that fit your search criteria in the database.
<class 'pystac.item_collection.ItemCollection'>


In [72]:
## STACK COLLECTION ITEMS

# create the stack
stack = stackstac.stack(items, 
                        epsg=proj.ext(items[0]).epsg, 
                        assets=['B04','B03','B02', 'B08', 'B11'],
                        bounds_latlon=bounds_latlon, 
                        resolution=10,
                        chunksize=(1, 1, 4096, 4096),
                        )

# mask_edge = (stack == 0).all(dim='band')
# stack = stack.where(~mask_edge)
# stack[1].values
stack.time[1:10]



<xarray.DataArray 'time' (time: 9)> Size: 72B
array(['2019-05-01T14:59:21.024000000', '2019-05-02T15:18:09.024000000',
       '2019-05-02T15:18:09.024000000', '2019-05-04T15:09:21.024000000',
       '2019-05-04T15:09:21.024000000', '2019-05-05T15:28:19.024000000',
       '2019-05-05T15:28:19.024000000', '2019-05-06T15:00:19.024000000',
       '2019-05-06T15:00:19.024000000'], dtype='datetime64[ns]')
Coordinates: (12/35)
  * time                                     (time) datetime64[ns] 72B 2019-0...
    id                                       (time) <U54 2kB 'S2A_MSIL2A_2019...
    sat:relative_orbit                       (time) int64 72B 125 68 ... 125 125
    s2:mean_solar_azimuth                    (time) float64 72B 176.4 ... 176.5
    s2:snow_ice_percentage                   (time) float64 72B 100.0 ... 99.96
    s2:medium_proba_clouds_percentage        (time) float64 72B 7.4e-05 ... 0.0
    ...                                       ...
    s2:reflectance_conversion_factor         (time) float64 72B 0.9872 ... 0....
    eo:cloud_cover                           (time) float64 72B 0.000662 ... ...
    s2:water_percentage                      (time) float64 72B 0.001781 ... ...
    instruments                              <U3 12B 'msi'
    s2:not_vegetated_percentage              (time) float64 72B 0.0 0.0 ... 0.0
    epsg                                     int64 8B 32622

In [83]:
# Daily average (one value per calendar day)
daily = stack.resample(time="D").mean("time", keep_attrs=True)

daily



<xarray.DataArray 'stackstac-48f3a657ff034465fbc430e04f6c0418' (time: 152,
                                                                band: 5,
                                                                y: 660, x: 660)> Size: 3GB
dask.array<where, shape=(152, 5, 660, 660), dtype=float64, chunksize=(1, 1, 660, 660), chunktype=numpy.ndarray>
Coordinates: (12/18)
  * band                                     (band) <U3 60B 'B04' ... 'B11'
  * x                                        (x) float64 5kB 5.987e+05 ... 6....
  * y                                        (y) float64 5kB 7.674e+06 ... 7....
  * time                                     (time) datetime64[ns] 1kB 2019-0...
    constellation                            <U10 40B 'Sentinel 2'
    s2:saturated_defective_pixel_percentage  float64 8B 0.0
    ...                                       ...
    gsd                                      (band) float64 40B 10.0 ... 20.0
    title                                    (band) <U26 520B 'Band 4 - Red -...
    common_name                              (band) <U6 120B 'red' ... 'swir16'
    center_wavelength                        (band) float64 40B 0.665 ... 1.61
    full_width_half_max                      (band) float64 40B 0.038 ... 0.143
    epsg                                     int64 8B 32622
Attributes:
    spec:        RasterSpec(epsg=32622, bounds=(598740, 7667540, 605340, 7674...
    crs:         epsg:32622
    transform:   | 10.00, 0.00, 598740.00|\n| 0.00,-10.00, 7674140.00|\n| 0.0...
    resolution:  10

In [96]:
## FUNCTION FOR GENERATE A STACK FOR A GIVEN LOCATION ACROSS A DATE RANGE, EACH DAY

def timestack_gen(centroid, band_names, bbox_size=6000, time_range='2019-05-01/2019-09-30', normalize=True, pull_to_mem=False):
    
    # SEARCH IMAGERY CATALOG FOR ITEMS MATCHING LOCATION AND DATE RANGE
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bounds_latlon,
        datetime=time_range,
        query={"eo:cloud_cover": {"lt": 100}} )
    items = search.item_collection()
    
    # CREATE THE STACK
    stack = stackstac.stack(items, 
                            epsg=proj.ext(items[0]).epsg, 
                            assets=['B04','B03','B02', 'B08', 'B11'],
                            bounds_latlon=bounds_latlon, 
                            resolution=10,
                            chunksize=(1, 1, 4096, 4096),
                            )
    
    # SAMPLE DAILY, PRESERVING CLOUD COVER IN THE METADATA
    stack_daily = stack.resample(time="D").mean("time", keep_attrs=True)
    cc_da = xr.DataArray(
        np.array([item.properties["eo:cloud_cover"] for item in items]),
        coords={"time": stack.time},
        dims=["time"])
    start, end = time_range.split("/")
    full_days = pd.date_range(start, end, freq="D")
    stack_daily = stack_daily.reindex(time=full_days, fill_value=np.nan)
    daily_cc = cc_da.resample(time="D").max()
    daily_cc = daily_cc.reindex(time=full_days, fill_value=np.nan)
    stack_daily = stack_daily.assign_coords(eo_cloud_cover=daily_cc)
    # print(f"stack_daily shape: {stack_daily.shape}")
    
    # IF NORMALIZING
    if normalize:
        def combo_scaler(x, range_max=1):
            median_x = np.nanmedian(x)
            iqr_x = np.nanpercentile(x,75) - np.nanpercentile(x,25)
            robust_x = ((x-median_x)/iqr_x)
            return ((robust_x - np.nanmin(robust_x)) / (np.nanmax(robust_x) - np.nanmin(robust_x))) * range_max

        stack_rechunk = stack_daily.chunk({'y': -1, 'x': -1})
        stack_daily = xr.apply_ufunc(
            combo_scaler,            # combo_scaler function
            stack_rechunk,                   # DataArray stack
            input_core_dims=[['y','x']],
            output_core_dims=[['y','x']],
            vectorize=True,          # loop over time & band dims
            dask='parallelized',
            output_dtypes=[float],
            kwargs={'range_max':1},
            keep_attrs=True)

    # CROP TILES TO DESIRED SIZE
    tile_size = 512 #[pix]
    pix_res = 10 #[m/pix]
    buffer = tile_size * pix_res/2 #[m]
    x_utm, y_utm = pyproj.Proj(stack.crs)(*centroid)
    timestack = stack_daily.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
    
    # PULL STACK INTO MEMORY
    if pull_to_mem:
        stack_to_pull = timestack
        print(f"pulling stack into memory, shape will be: {tile_stack.shape}")
        with dask.diagnostics.ProgressBar():
            timestack_mem = stack_to_pull.compute()
        print(f"shape of stack in memory: {timestack_mem.shape}")
        return timestack_mem
    else:
        print(f"timestack (not in mem) shape: {timestack.shape}\n")
        return timestack
    
    
    


In [99]:
## CALL FUNCTION TO GENERATE TILE STACK


# Define centroid and bounding box
centroid = (gdf.iloc[idx_lake].centroid_x, gdf.iloc[idx_lake].centroid_y)
bounds_latlon = bounds_latlon_around(*centroid, side_m=6000)

# Specify time range
time_range = '2019-05-01/2019-09-30'

# Specify band names
band_names = ["B04",  # red (665 nm)
              "B03",  # green (560 nm)
              "B02",  # blue (490 nm)
              "B08",  # NIR (842 nm)
              "B11"]  # SWIR1 (1610 nm)

timestack = timestack_gen(centroid, band_names, bbox_size=6000, time_range='2019-05-01/2019-09-30', normalize=True, pull_to_mem=True)


pulling stack into memory, shape will be: (153, 5, 512, 512)
[########################################] | 100% Completed | 52.45 s
shape of stack in memory: (153, 5, 512, 512)


In [100]:
## LOOP OVER LOCATIONS (LAKENUMS) AND BUILD FULL ARRAY

# READ IN THE LAKE INFORMATION .geojson FILE
gdf = gpd.read_file("/home/jupyter/repos/lake-vision/sandbox/dunmire_cw2019_shuttle.geojson")
# gdf = gdf.to_crs(tiles_full.rio.crs)  # make sure it’s in the same CRS as the imagery in the stack

normalize = False

# LOOP OVER LAKE LOCATIONS
# for idx_lake in range(len(gdf)):
for idx_lake in range(1):
    
    print(f"searching for imagery for lakenum {gdf.lakenum.iloc[idx_lake]}")
    
    # BOUNDING BOX AROUND LAKE CENTROID
    centroid = (gdf.iloc[idx_lake].centroid_x, gdf.iloc[idx_lake].centroid_y)
    bounds_latlon = bounds_latlon_around(*centroid, side_m=6000)
    
    # SPECIFY TIME RANGE
    time_range = '2019-05-01/2019-09-30'
    
    # SPECIFY IMAGERY BANDS
    band_names = ["B04",  # red (665 nm)
                  "B03",  # green (560 nm)
                  "B02",  # blue (490 nm)
                  "B08",  # NIR (842 nm)
                  "B11"]  # SWIR1 (1610 nm)
    
    # CALL FUNCTION TO GENERATE TIMESTACK
    timestack = timestack_gen(centroid, band_names, bbox_size=6000, time_range='2019-05-01/2019-09-30', normalize=True, pull_to_mem=True)
    
    # SAVE STACK TO DISK OR MAYBE NEED TO STACK THESE ALL TOGETHER ACROSS LAKES INTO ONE FILE???
    
    
    




searching for imagery for lakenum 2019cw_1
pulling stack into memory, shape will be: (153, 5, 512, 512)
[########################################] | 100% Completed | 50.36 s
shape of stack in memory: (153, 5, 512, 512)


In [11]:
# ## FUNCTION TO FILTER LIST OF ITEMS TO SELECT THE BEST ITEM PER DATE BASED ON MAXIMUM COVERAGE 

# def best_items_per_date(items, bbox):
#     """
#     From a flat list of pystac.Item, return one Item per sensing-date—
#     namely, the tile whose bbox overlaps the given AOI bbox the most.
    
#     Parameters
#     ----------
#     items : Sequence[pystac.Item]
#         your full STAC Item list
#     bbox : tuple(float, float, float, float)
#         (minx, miny, maxx, maxy) in lon/lat
    
#     Returns
#     -------
#     List[pystac.Item]
#         one “best‐overlap” Item per unique sensing date
#     """
#     aoi = box(*bbox)
    
#     # 1) bucket items by date
#     items_by_date = defaultdict(list)
#     for item in items:
#         dt = item.datetime.date()
#         items_by_date[dt].append(item)
    
#     # 2) pick best per date
#     best = []
#     for dt, group in items_by_date.items():
#         # compute overlap for each tile of that date
#         overlaps = {
#             item: aoi.intersection(box(*item.bbox)).area
#             for item in group
#         }
#         # choose the one with max intersection area
#         best_item = max(overlaps, key=overlaps.get)
#         best.append(best_item)
        
#     best_collection = ItemCollection(best)
    
#     return best_collection



In [6]:
# ## FUNCTION TO GET LISTS OF DATES WITH CLEAR TILES, CLOUDY TILES, AND BLANK TILES AS WELL AS A VECTOR OF 0,1,2 REPRESENTING CLEAR,CLOUDY,BLANK TILES IN THE SEQUENCE
# # dates_cleartiles = [item.datetime.date().isoformat() for item in items]

# def dates_clearcloudyblank(cloud_limit, collection_items):

#     # parse calendar range
#     start_str, end_str = time_range.split("/")
#     start_date = datetime.datetime.fromisoformat(start_str).date()
#     end_date   = datetime.datetime.fromisoformat(end_str).date()
#     n_days   = (end_date - start_date).days + 1
    
#     # build ordered list of all dates in calendar range
#     all_dates = [
#         (start_date + datetime.timedelta(days=i)).isoformat()
#         for i in range(n_days)
#     ]

#     # split colleted dates into clear/cloudy
#     dates_clear  = []
#     dates_cloudy = []
#     for item in items:
#         d = item.datetime.date().isoformat()
#         if item.properties["eo:cloud_cover"] < cloud_limit:
#             dates_clear.append(d)
#         else:
#             dates_cloudy.append(d)

#     # compute missing dates (will later become blank tiles)
#     seen = set(dates_clear) | set(dates_cloudy)
#     dates_blank = [d for d in all_dates if d not in seen]

#     # build ccb_vec with entries of 0 (clear day), 1 (cloudy day), 2 (missing/blank day)
#     ccb_list = ccb_vec = [
#         0 if d in dates_clear
#         else 1 if d in dates_cloudy
#         else 2
#         for d in all_dates
#     ]
#     ccb = np.array(ccb_list)

#     # raise warning if there are repeated dates


#     return dates_clear, dates_cloudy, dates_blank, ccb


In [54]:
## APPLY FUNCTION TO GET LISTS OF DATES WITH CLEAR TILES, CLOUDY TILES, AND BLANK TILES

cloud_limit = 50 #[percent]
dates_clear, dates_cloudy, dates_blank, ccb = dates_clearcloudyblank(cloud_limit, best_items)

print(f"number of clear dates: {len(dates_clear)}")
print(f"number of cloudy dates: {len(dates_cloudy)}")
print(f"number of missing dates: {len(dates_blank)}")
print(f"total dates: {len(dates_cloudy)+len(dates_clear)+len(dates_blank)}")
print(f"len(ccb) = {len(ccb)}")


number of clear dates: 193
number of cloudy dates: 34
number of missing dates: 33
total dates: 260
len(ccb) = 153


In [44]:
## STACK COLLECTION ITEMS

# create the stack
stack = stackstac.stack(items, 
                        epsg=proj.ext(items[0]).epsg, 
                        assets=['B04','B03','B02', 'B08', 'B11'],
                        bounds_latlon=bounds_latlon, 
                        resolution=10,
                        chunksize=(1, 1, 4096, 4096),
                        )

# mask_edge = (stack == 0).all(dim='band')
# stack = stack.where(~mask_edge)
stack[1].values


array([[[ 9864.,  9904.,  9896., ...,  9744.,  9792.,  9800.],
        [ 9872.,  9880.,  9864., ...,  9752.,  9768.,  9664.],
        [ 9856.,  9856.,  9872., ...,  9704.,  9736.,  9696.],
        ...,
        [ 9120.,  8896.,  8768., ...,  9312.,  9264.,  9256.],
        [ 9464.,  9064.,  8560., ...,  9264.,  9176.,  9264.],
        [ 9208.,  9040.,  8696., ...,  9320.,  9280.,  9312.]],

       [[ 9992., 10040.,  9968., ...,  9880.,  9904.,  9944.],
        [ 9968.,  9984.,  9984., ...,  9944.,  9944.,  9856.],
        [ 9992.,  9976.,  9928., ...,  9880.,  9888.,  9856.],
        ...,
        [ 9344.,  8872.,  9016., ...,  9472.,  9496.,  9360.],
        [ 9568.,  9152.,  8856., ...,  9440.,  9320.,  9288.],
        [ 9216.,  9168.,  9016., ...,  9464.,  9432.,  9424.]],

       [[ 9968.,  9992.,  9976., ...,  9968.,  9960.,  9960.],
        [ 9928., 10008.,  9976., ...,  9968.,  9976.,  9848.],
        [ 9976.,  9952.,  9920., ...,  9880.,  9936.,  9928.],
        ...,
        [ 93

In [7]:
# FIRST NORMALIZE EACH IMAGE IN STACK BEFORE TAKING WEEKLY RESAMPLE
def combo_scaler(x, range_max=1):
    median_x = np.nanmedian(x)
    iqr_x = np.nanpercentile(x,75) - np.nanpercentile(x,25)
    robust_x = ((x-median_x)/iqr_x)
    
    return ((robust_x - np.nanmin(robust_x)) / (np.nanmax(robust_x) - np.nanmin(robust_x))) * range_max


stack_rechunk = stack.chunk({'y': -1, 'x': -1})

stack_scaled = xr.apply_ufunc(
    combo_scaler,            # combo_scaler function
    stack_rechunk,                   # DataArray stack
    input_core_dims=[['y','x']],
    output_core_dims=[['y','x']],
    vectorize=True,          # loop over time & band dims
    dask='parallelized',
    output_dtypes=[float],
    kwargs={'range_max':1},
    keep_attrs=True
)


NameError: name 'stack' is not defined

In [59]:
# ## DO WEEKLY AVERAGING

# # NOW TAKE WEEKLY RESAMPLE
# weekly = stack_scaled.resample(time="W").mean("time", keep_attrs=True)
# # weekly

# # weekly_scaled = stack_scaled.resample(time="W").mean("time", keep_attrs=True)
# # weekly_scaled


In [16]:
## SPECIFY TILE OF SPECIFIED PIXEL DIMENSION (e.g., 512x512) TO CROP OUT OF THE LARGER rgb STACK

tile_size = 512 #[pix]
pix_res = 10 #[m/pix]
buffer = tile_size * pix_res/2 #[m]
x_utm, y_utm = pyproj.Proj(stack.crs)(*centroid)

tile_stack = stack.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
tile_stack

# tile_stack_scaled = stack_scaled.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
# tile_stack_scaled

# weekly_stack = weekly.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
# weekly_stack

# weekly_stack_scaled = weekly_scaled.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
# weekly_stack_scaled


<xarray.DataArray 'stackstac-edfa7a9ed0e3c73520bcc5ef794222e1' (time: 121,
                                                                band: 5,
                                                                y: 512, x: 512)> Size: 1GB
dask.array<getitem, shape=(121, 5, 512, 512), dtype=float64, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>
Coordinates: (12/44)
  * time                                     (time) datetime64[ns] 968B 2019-...
    id                                       (time) <U54 26kB 'S2A_MSIL2A_201...
  * band                                     (band) <U3 60B 'B04' ... 'B11'
  * x                                        (x) float64 4kB 5.584e+05 ... 5....
  * y                                        (y) float64 4kB 7.627e+06 ... 7....
    s2:nodata_pixel_percentage               (time) float64 968B 54.92 ... 2....
    ...                                       ...
    title                                    (band) <U26 520B 'Band 4 - Red -...
    gsd                                      (band) float64 40B 10.0 ... 20.0
    common_name                              (band) <U6 120B 'red' ... 'swir16'
    center_wavelength                        (band) float64 40B 0.665 ... 1.61
    full_width_half_max                      (band) float64 40B 0.038 ... 0.143
    epsg                                     int64 8B 32622
Attributes:
    spec:        RasterSpec(epsg=32622, bounds=(557640, 7621150, 564230, 7627...
    crs:         epsg:32622
    transform:   | 10.00, 0.00, 557640.00|\n| 0.00,-10.00, 7627740.00|\n| 0.0...
    resolution:  10

In [18]:
## PULL IN DESIRED STACK

stack_to_pull = tile_stack

with dask.diagnostics.ProgressBar():
    stack_in_mem = stack_to_pull.compute()
print(f"shape of stack in memory: {stack_in_mem.shape}")

[########################################] | 100% Completed | 32.47 s
shape of stack in memory: (121, 5, 512, 512)


In [20]:
# ## FILL MISSING DAYS WITH BLANK TILES

# # 1) parse calendar range
# start_str, end_str = time_range.split("/")
# full_dates = pd.date_range(
#     start=start_str,
#     end=end_str,
#     freq="D"
# ).normalize()            # midnight‐based daily index

# # 2) normalize your existing times to dates
# tiles_date = stack_in_mem.copy().assign_coords(
#     time = stack_in_mem.time.dt.floor("D")
# )

# # 3) find which dates are missing
# existing = pd.DatetimeIndex(tiles_date.time.values)
# missing = full_dates.difference(existing)

# # 4) build a blank‐tile template for one date
# template = tiles_date.isel(time=0) * 0

# # 5) expand that template into a DataArray for all missing dates
# blank = template.expand_dims(time=missing)  # shape: (len(missing), band, y, x)

# # 6) concat real + blank, then sort by time
# tiles_full = xr.concat([tiles_date, blank], dim="time").sortby("time")

# # 7) (optional) restore any original hourly times on the real tiles
# #     — or just leave them floored to midnight if daily is fine.

# tiles_full
# # now tiles_full.time == full_dates, and any originally‐missing day is a zero‐tile


<xarray.DataArray 'stackstac-edfa7a9ed0e3c73520bcc5ef794222e1' (time: 153,
                                                                band: 5,
                                                                y: 512, x: 512)> Size: 2GB
array([[[[ 9656.,  9680.,  9616., ...,  9472.,  9608.,  9784.],
         [ 9736.,  9720.,  9496., ...,  9416.,  9616.,  9760.],
         [ 9752.,  9640.,  9432., ...,  9728.,  9856.,  9728.],
         ...,
         [ 9616.,  9568.,  9600., ...,  9584.,  9568.,  9648.],
         [ 9592.,  9640.,  9712., ...,  9584.,  9640.,  9672.],
         [ 9600.,  9640.,  9632., ...,  9600.,  9640.,  9712.]],

        [[ 9872.,  9904.,  9776., ...,  9824.,  9904.,  9944.],
         [ 9840.,  9872.,  9728., ...,  9768.,  9920.,  9920.],
         [ 9856.,  9784.,  9616., ...,  9960., 10184., 10080.],
         ...,
         [ 9768.,  9808.,  9808., ...,  9752.,  9816.,  9824.],
         [ 9720.,  9768.,  9784., ...,  9736.,  9752.,  9808.],
         [ 9688.,  9768.,  9760., ...,  9808.,  9784.,  9792.]],

        [[ 9856.,  9936.,  9776., ...,  9784.,  9920., 10072.],
         [ 9912.,  9920.,  9816., ...,  9672., 10032., 10056.],
         [ 9968.,  9744.,  9608., ...,  9984., 10176., 10120.],
         ...,
...
         ...,
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.]],

        [[    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         ...,
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.]],

        [[    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         ...,
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.],
         [    0.,     0.,     0., ...,     0.,     0.,     0.]]]])
Coordinates: (12/44)
    id                                       (time) <U54 33kB 'S2A_MSIL2A_201...
  * band                                     (band) <U3 60B 'B04' ... 'B11'
  * x                                        (x) float64 4kB 5.584e+05 ... 5....
  * y                                        (y) float64 4kB 7.627e+06 ... 7....
    s2:nodata_pixel_percentage               (time) float64 1kB 54.92 ... 54.92
    s2:datastrip_id                          (time) <U64 39kB 'S2A_OPER_MSI_L...
    ...                                       ...
    gsd                                      (band) float64 40B 10.0 ... 20.0
    common_name                              (band) <U6 120B 'red' ... 'swir16'
    center_wavelength                        (band) float64 40B 0.665 ... 1.61
    full_width_half_max                      (band) float64 40B 0.038 ... 0.143
    epsg                                     int64 8B 32622
  * time                                     (time) datetime64[ns] 1kB 2019-0...
Attributes:
    spec:        RasterSpec(epsg=32622, bounds=(557640, 7621150, 564230, 7627...
    crs:         epsg:32622
    transform:   | 10.00, 0.00, 557640.00|\n| 0.00,-10.00, 7627740.00|\n| 0.0...
    resolution:  10

In [1]:
plt.imshow(stack_in_mem[15,0,])



NameError: name 'plt' is not defined

In [15]:
## TO OPEN .zarr FROM DISK

stack_name = "tiles_wk"
ds = xr.open_zarr(f"/home/jupyter/{stack_name}.zarr", chunks="auto")
tiles_reload = ds[f'{stack_name}']
tiles_reload

<xarray.DataArray 'tiles_wk' (time: 22, band: 5, y: 2048, x: 2048)> Size: 4GB
dask.array<open_dataset-tiles_wk, shape=(22, 5, 2048, 2048), dtype=float64, chunksize=(3, 1, 256, 512), chunktype=numpy.ndarray>
Coordinates: (12/19)
  * band                                     (band) <U3 60B 'B04' ... 'B11'
    center_wavelength                        (band) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    common_name                              (band) <U6 120B dask.array<chunksize=(5,), meta=np.ndarray>
    constellation                            <U10 40B ...
    epsg                                     int64 8B ...
    full_width_half_max                      (band) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    ...                                       ...
    s2:product_type                          <U7 28B ...
    s2:saturated_defective_pixel_percentage  float64 8B ...
  * time                                     (time) datetime64[ns] 176B 2019-...
    title                                    (band) <U26 520B dask.array<chunksize=(5,), meta=np.ndarray>
  * x                                        (x) float64 16kB 5.507e+05 ... 5...
  * y                                        (y) float64 16kB 7.635e+06 ... 7...
Attributes:
    crs:         epsg:32622
    resolution:  10
    transform:   [10.0, 0.0, 549970.0, 0.0, -10.0, 7635420.0, 0.0, 0.0, 1.0]